In [ ]:
# @title Instalar Apollo (ejecutar una vez por sesion) { display-mode: "form" }
# set -e: si git, pip o la descarga de pesos fallan, la celda se corta con el
# error a la vista. Sin esto, un fallo deja el entorno a medias y el error
# recien aparece al intentar procesar. Por eso no se usa %%capture.
%%bash
set -e

REPO="https://github.com/rcruzotiniano/Apollo_Mod.git"
DESTINO="/content/Apollo_Mod"

# git-lfs es imprescindible: los pesos de model/ viven en LFS.
if ! command -v git-lfs >/dev/null 2>&1; then
  echo "==> Instalando git-lfs..."
  apt-get update -qq
  apt-get install -y -qq git-lfs
fi
git lfs install --skip-repo

# Idempotente: si la sesion ya clono el repo, se actualiza en lugar de fallar.
if [ ! -d "$DESTINO/.git" ]; then
  echo "==> Clonando Apollo..."
  rm -rf "$DESTINO"
  git clone -q "$REPO" "$DESTINO"
else
  echo "==> El repo ya estaba clonado: actualizando..."
  git -C "$DESTINO" pull -q --ff-only
fi

# Solo falta ml_collections: Colab ya trae torch con CUDA, librosa,
# soundfile y scipy. NO instalar requirements.txt aca: fija torch y numpy a
# versiones que degradarian el entorno de Colab.
echo "==> Instalando dependencias que faltan..."
pip install -q ml_collections

echo "==> Descargando los pesos (314 MB, tarda un par de minutos)..."
git -C "$DESTINO" lfs pull
echo "==> Listo. Pesos descargados:"
ls -lh "$DESTINO/model"


In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title Verificar el entorno antes de procesar { display-mode: "form" }
import pathlib
import sys

import torch

PROYECTO = pathlib.Path("/content/Apollo_Mod")
if not (PROYECTO / "inference.py").is_file():
    raise SystemExit(
        "Falta /content/Apollo_Mod. Ejecuta primero la celda de instalacion."
    )
sys.path.insert(0, str(PROYECTO))

# Sin GPU, procesar una carpeta de audios es inviable (horas en vez de
# minutos). Mejor enterarse ahora que a mitad del trabajo.
if not torch.cuda.is_available():
    raise SystemExit(
        "No hay GPU. Activala en Entorno de ejecucion -> Cambiar tipo de "
        "entorno -> T4 GPU y volve a ejecutar esta celda."
    )
print("GPU:", torch.cuda.get_device_name(0))

# Comprueba que los 4 checkpoints esten bajados y que el motor responda.
import inference

faltan = []
for nombre, (etiqueta, ckpt, config, sufijo) in inference.MODEL_CONFIGS.items():
    if not inference.resolve_project_path(ckpt).is_file():
        faltan.append(nombre)
if faltan:
    raise SystemExit(
        "Faltan checkpoints de: "
        + ", ".join(faltan)
        + ". Ejecuta en una celda: !git -C /content/Apollo_Mod lfs pull"
    )
print("Modelos:", ", ".join(sorted(inference.MODEL_CONFIGS)))
print("Entorno OK. Podes procesar.")


In [ ]:
# @title Procesar { display-mode: "form" }
# Modelo de Apollo a aplicar (ver la tabla de modelos en el README).
MODEL = "normal" #@param ["normal", "vocal", "vocal_v2", "universal"]
# Cutoff del filtro pasabajos de entrada, en Hz. El campo es editable:
# elegi cualquier valor de 1000 a 24000 (por debajo de 22050). 0 = sin cutoff.
CUTOFF_HZ = 16000 #@param [0, 8000, 11000, 11025, 12000, 14000, 15000, 16000, 16500, 17000, 18000, 20000, 22000]
# Carpeta a procesar: recursiva, incluye subcarpetas.
INPUT_DIR = "/content/drive/MyDrive/Apollo" #@param {type:"string"}
# Carpeta de salida. Vacio = guardar junto a los originales.
OUTPUT_DIR = "" #@param {type:"string"}

%cd /content/Apollo_Mod
!python inference.py --input_dir "{INPUT_DIR}" --model "{MODEL}" --cutoff {CUTOFF_HZ} --output_dir "{OUTPUT_DIR}"
